# Lesson 1.2 · Evaluating a regression model

A model is not good because it has a high R² on the data you trained it with: it is good if it generalizes. Here you see the
three fundamental metrics, the role of the baseline and cross-validation, all applied to a real dataset.

**What you will learn**

- To compute MAE, RMSE and R² by hand and with scikit-learn, without swapping the order of the arguments.
- To judge a number by comparing it with a baseline that knows nothing.
- To estimate the uncertainty of the performance with cross-validation.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

## The dataset

California housing: 20,640 districts, 8 features (median income, age of the houses, latitude...) and as target the
median house value, in hundreds of thousands of dollars. On first use scikit-learn downloads it and
keeps it in the `data/` folder.

In [ ]:
X, y = ds.california()
print(X.shape)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
X.head(3)

## The three metrics

```
MAE  = mean of |y − ŷ|                     same unit as the target; every error counts in proportion
RMSE = square root of the mean of (y − ŷ)²  same unit as the target; large errors count more
R²   = 1 − SSR / SST                        fraction of the error of the mean that the model removes
```

where SSR is the sum of squares of the model residuals and SST the sum of squares of the deviations from the mean of y.

In [ ]:
model = LinearRegression().fit(X_tr, y_tr)
pred = model.predict(X_te)

mae = np.abs(y_te - pred).mean()
rmse = np.sqrt(((y_te - pred) ** 2).mean())
r2 = 1 - ((y_te - pred) ** 2).sum() / ((y_te - y_te.mean()) ** 2).sum()
print(f"by hand: MAE {mae:.4f} | RMSE {rmse:.4f} | R2 {r2:.4f}")

print(f"sklearn: MAE {mean_absolute_error(y_te, pred):.4f} | "
      f"RMSE {mean_squared_error(y_te, pred, squared=False):.4f} | R2 {r2_score(y_te, pred):.4f}")

> **Common mistake.** All the functions in `sklearn.metrics` want the **true** values first and then the predictions:
> `r2_score(y_true, y_pred)`. With MAE and MSE swapping the arguments does not change the result because they are symmetric;
> with R² it does, and it is a silent bug (you will hunt for it in exercise E2).
>
> Version note: `squared=False` computes the root in scikit-learn 1.1. In recent versions it has been replaced
> by `root_mean_squared_error`.

## The baseline: what a model that knows nothing does

An RMSE of 0.73 means nothing until you know what the dumbest possible model scores, the one that always predicts
the mean of the training set. R² is exactly this comparison made explicit.

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
pred_base = base.predict(X_te)

print(f"baseline: RMSE {mean_squared_error(y_te, pred_base, squared=False):.4f} | R2 {r2_score(y_te, pred_base):.4f}")
print(f"model   : RMSE {rmse:.4f} | R2 {r2:.4f}")

The R² of the baseline on the test set is slightly **negative**, not zero: the mean of the training set does not match
that of the test set, so it does a bit worse than predicting the test mean. A negative R² is not a calculation
error: it means the model does worse than the mean.

## Cross-validation

A single split gives you one number. Cross-validation gives you five, and their spread tells you how far you can
trust it. In scikit-learn the metrics "to minimize" are passed with the sign flipped (`neg_...`), because the
internal convention is that the highest score is the best.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
scores = -cross_val_score(LinearRegression(), X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error")

print("RMSE per fold:", scores.round(4))
print(f"mean RMSE {scores.mean():.4f} +/- {scores.std():.4f}")

## A useful indicator: the RMSE / MAE ratio

If the errors were all the same size the two values would coincide. The RMSE is always greater than or equal
to the MAE, and the gap between the two tells you how heavy the tails of the errors are.

In [ ]:
print("RMSE / MAE:", round(rmse / mae, 2))
cap = 5.0     # in this dataset the price is capped at 5.0: the most expensive houses are all "5"
capped = (y_te >= cap).to_numpy()
print("share of houses with a capped price:", round(capped.mean(), 3))
print("RMSE on the capped ones:", round(np.sqrt(((y_te[capped] - pred[capped]) ** 2).mean()), 3),
      "| on the others:", round(np.sqrt(((y_te[~capped] - pred[~capped]) ** 2).mean()), 3))

## Check your understanding

<details><summary>An R² of 0.60 on the training set and 0.58 in cross-validation: overfitting?</summary>

No. The gap is small, so the model generalizes almost as well as it fits. Overfitting shows up as a high training R² and a much lower validation score.

</details>

<details><summary>Why shouldn't the test set be used to choose between two models?</summary>

Because choosing the best one by looking at the test set turns it into part of the selection process: the score you will report for the chosen model is optimistically biased. To choose you use cross-validation on the training set, and you touch the test set only once at the end.

</details>